# Description

(Please, take a look at the README.md file in this directory for instructions on how to run this notebook)

This notebook compiles information about the GWAS and TWAS for a particular cohort. For example, the set of GWAS variants, variance of predicted expression of genes, etc.

It has specicfic parameters for papermill (see under `Settings` below).

This notebook is not directly run. See README.md.

# Modules

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

import conf
from entity import Gene

# Settings

In [3]:
# a cohort name (it could be something like UK_BIOBANK, etc)
COHORT_NAME = None

# reference panel such as 1000G or GTEX_V8
REFERENCE_PANEL = None

# predictions models such as MASHR or ELASTIC_NET
EQTL_MODEL = None

# a string with a path pointing to an imputed GWAS
GWAS_FILE = None

# a string with a path pointing where S-PrediXcan results (tissue-specific are located
SPREDIXCAN_FOLDER = None

# an f-string with one placeholder {tissue}
SPREDIXCAN_FILE_PATTERN = None

# a string with a path pointing to an S-MultiXcan result
SMULTIXCAN_FILE = None

In [4]:
# Parameters
PHENOPLIER_NOTEBOOK_FILEPATH = (
    "nbs/15_gsa_gls/07-compile_gwas_snps_and_twas_genes.ipynb"
)
COHORT_NAME = "ukbb_eur"
REFERENCE_PANEL = "GTEX_V8"
EQTL_MODEL = "MASHR"
GWAS_FILE = "/home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/final_imputed_gwas/random.pheno0.glm-imputed.txt.gz"
SPREDIXCAN_FOLDER = "/home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/twas/spredixcan/"
SPREDIXCAN_FILE_PATTERN = "random.pheno0-gtex_v8-mashr-{tissue}.csv"
SMULTIXCAN_FILE = "/home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/twas/smultixcan/random.pheno0-gtex_v8-mashr-smultixcan.txt"


In [5]:
assert COHORT_NAME is not None and len(COHORT_NAME) > 0, "A cohort name must be given"

COHORT_NAME = COHORT_NAME.lower()
display(f"Cohort name: {COHORT_NAME}")

'Cohort name: ukbb_eur'

In [6]:
assert (
    REFERENCE_PANEL is not None and len(REFERENCE_PANEL) > 0
), "A reference panel must be given"

display(f"Reference panel: {REFERENCE_PANEL}")

'Reference panel: GTEX_V8'

In [7]:
assert GWAS_FILE is not None and len(GWAS_FILE) > 0, "A GWAS file path must be given"
GWAS_FILE = Path(GWAS_FILE).resolve()
assert GWAS_FILE.exists(), "GWAS file does not exist"

display(f"GWAS file path: {str(GWAS_FILE)}")

'GWAS file path: /home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/final_imputed_gwas/random.pheno0.glm-imputed.txt.gz'

In [8]:
assert (
    SPREDIXCAN_FOLDER is not None and len(SPREDIXCAN_FOLDER) > 0
), "An S-PrediXcan folder path must be given"
SPREDIXCAN_FOLDER = Path(SPREDIXCAN_FOLDER).resolve()
assert SPREDIXCAN_FOLDER.exists(), "S-PrediXcan folder does not exist"

display(f"S-PrediXcan folder path: {str(SPREDIXCAN_FOLDER)}")

'S-PrediXcan folder path: /home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/twas/spredixcan'

In [9]:
assert (
    SPREDIXCAN_FILE_PATTERN is not None and len(SPREDIXCAN_FILE_PATTERN) > 0
), "An S-PrediXcan file pattern must be given"
assert (
    "{tissue}" in SPREDIXCAN_FILE_PATTERN
), "S-PrediXcan file pattern must have a '{tissue}' placeholder"

display(f"S-PrediXcan file template: {SPREDIXCAN_FILE_PATTERN}")

'S-PrediXcan file template: random.pheno0-gtex_v8-mashr-{tissue}.csv'

In [10]:
assert (
    SMULTIXCAN_FILE is not None and len(SMULTIXCAN_FILE) > 0
), "An S-MultiXcan result file path must be given"
SMULTIXCAN_FILE = Path(SMULTIXCAN_FILE).resolve()
assert SMULTIXCAN_FILE.exists(), "S-MultiXcan result file does not exist"

display(f"S-MultiXcan file path: {str(SMULTIXCAN_FILE)}")

'S-MultiXcan file path: /home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/twas/smultixcan/random.pheno0-gtex_v8-mashr-smultixcan.txt'

In [11]:
assert (
    EQTL_MODEL is not None and len(EQTL_MODEL) > 0
), "A prediction/eQTL model must be given"

display(f"eQTL model: {EQTL_MODEL}")

'eQTL model: MASHR'

In [12]:
OUTPUT_DIR_BASE = (
    conf.RESULTS["GLS"]
    / "gene_corrs"
    / "cohorts"
    / COHORT_NAME
    / REFERENCE_PANEL.lower()
    / EQTL_MODEL.lower()
)

OUTPUT_DIR_BASE.mkdir(parents=True, exist_ok=True)

display(f"Using output dir base: {OUTPUT_DIR_BASE}")

'Using output dir base: /home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/gene_corrs/cohorts/ukbb_eur/gtex_v8/mashr'

# Load MultiPLIER Z genes

In [13]:
multiplier_z_genes = pd.read_pickle(
    conf.MULTIPLIER["MODEL_Z_MATRIX_FILE"]
).index.tolist()

In [14]:
len(multiplier_z_genes)

6750

In [15]:
assert len(multiplier_z_genes) == len(set(multiplier_z_genes))

In [16]:
multiplier_z_genes[:5]

['GAS6', 'MMP14', 'DSP', 'MARCKSL1', 'SPARC']

# GWAS

In [17]:
gwas_file_columns = pd.read_csv(GWAS_FILE, sep="\t", nrows=2).columns

assert (
    "panel_variant_id" in gwas_file_columns
), "The GWAS file must be the final imputed one using the TWAS imputation tools with column 'panel_variant_id'"

assert (
    "pvalue" in gwas_file_columns
), "The GWAS file must be the final imputed one using the TWAS imputation tools with column 'pvalue'"

assert (
    "zscore" in gwas_file_columns
), "The GWAS file must be the final imputed one using the TWAS imputation tools with column 'zscore'"

In [18]:
gwas_data = pd.read_csv(
    GWAS_FILE,
    sep="\t",
    usecols=["panel_variant_id", "pvalue", "zscore"],
)

In [19]:
gwas_data.shape

(8842077, 3)

In [20]:
gwas_data.head()

,panel_variant_id,zscore,pvalue
0,chr1_13550_G_A_b38,-0.844665,0.398298
1,chr1_14671_G_C_b38,-0.456573,0.647978
2,chr1_14677_G_A_b38,0.823423,0.410268
3,chr1_14933_G_A_b38,0.803915,0.421446
4,chr1_16841_G_T_b38,0.076289,0.939189


In [21]:
gwas_data.dropna().shape

(8842077, 3)

In [22]:
# remove SNPs with no results
gwas_data = gwas_data.dropna()

In [23]:
gwas_data.shape

(8842077, 3)

## Save GWAS variants

In [24]:
gwas_data.head()

,panel_variant_id,zscore,pvalue
0,chr1_13550_G_A_b38,-0.844665,0.398298
1,chr1_14671_G_C_b38,-0.456573,0.647978
2,chr1_14677_G_A_b38,0.823423,0.410268
3,chr1_14933_G_A_b38,0.803915,0.421446
4,chr1_16841_G_T_b38,0.076289,0.939189


In [25]:
# in eMERGE's results, some values here are repeated (will be removed later by taking the unique set of variant IDs).
gwas_data["panel_variant_id"].is_unique

True

In [26]:
gwas_variants_ids_set = frozenset(gwas_data["panel_variant_id"])
list(gwas_variants_ids_set)[:5]

['chr3_3407685_T_C_b38',
 'chr6_134169817_A_AC_b38',
 'chr11_27202069_G_A_b38',
 'chr3_36846681_T_G_b38',
 'chr4_24476831_G_A_b38']

In [27]:
with open(OUTPUT_DIR_BASE / "gwas_variant_ids.pkl", "wb") as handle:
    pickle.dump(gwas_variants_ids_set, handle, protocol=pickle.HIGHEST_PROTOCOL)

# TWAS

## Available tissues for eQTL model

In [28]:
prediction_model_tissues = conf.PHENOMEXCAN["PREDICTION_MODELS"][
    f"{EQTL_MODEL}_TISSUES"
].split(" ")

In [29]:
len(prediction_model_tissues)

49

In [30]:
prediction_model_tissues[:5]

['Thyroid',
 'Artery_Aorta',
 'Heart_Atrial_Appendage',
 'Liver',
 'Heart_Left_Ventricle']

## S-MultiXcan results

In [31]:
smultixcan_results = pd.read_csv(
    SMULTIXCAN_FILE, sep="\t", usecols=["gene", "gene_name", "pvalue", "n", "n_indep"]
)

In [32]:
smultixcan_results.shape

(22519, 5)

In [33]:
smultixcan_results = smultixcan_results.dropna()

In [34]:
smultixcan_results.shape

(22515, 5)

In [35]:
smultixcan_results = smultixcan_results.assign(
    gene_id=smultixcan_results["gene"].apply(lambda g: g.split(".")[0])
)

In [36]:
smultixcan_results.head()

,gene,gene_name,pvalue,n,n_indep,gene_id
0,ENSG00000254416.5,RP11-347E10.1,0.000017,4.0,3.0,ENSG00000254416
1,ENSG00000271564.1,RP13-444H2.1,0.000300,1.0,1.0,ENSG00000271564
2,ENSG00000181788.3,SIAH2,0.000467,45.0,1.0,ENSG00000181788
3,ENSG00000168830.7,HTR1E,0.000522,18.0,3.0,ENSG00000168830
4,ENSG00000148734.7,NPFFR1,0.000618,22.0,6.0,ENSG00000148734


In [37]:
assert smultixcan_results["gene_id"].is_unique

### Get common genes with MultiPLIER

In [38]:
common_genes = set(multiplier_z_genes).intersection(
    set(smultixcan_results["gene_name"])
)

In [39]:
len(common_genes)

6451

In [40]:
sorted(list(common_genes))[:5]

['A2M', 'AAAS', 'AANAT', 'AARS', 'AARS2']

## Genes info

In [41]:
multiplier_gene_obj = {
    gene_name: Gene(name=gene_name)
    for gene_name in common_genes
    if gene_name in Gene.GENE_NAME_TO_ID_MAP
}

In [42]:
# delete common_genes, from now on, genes_info should be used for common genes
del common_genes

In [43]:
len(multiplier_gene_obj)

6451

In [44]:
assert multiplier_gene_obj["GAS6"].ensembl_id == "ENSG00000183087"

In [45]:
_gene_obj = list(multiplier_gene_obj.values())

genes_info = pd.DataFrame(
    {
        "name": [g.name for g in _gene_obj],
        "id": [g.ensembl_id for g in _gene_obj],
        "chr": [g.chromosome for g in _gene_obj],
        "band": [g.band for g in _gene_obj],
        "start_position": [g.get_attribute("start_position") for g in _gene_obj],
        "end_position": [g.get_attribute("end_position") for g in _gene_obj],
    }
)

In [46]:
genes_info = genes_info.assign(
    gene_length=genes_info.apply(
        lambda x: x["end_position"] - x["start_position"], axis=1
    )
)

In [47]:
genes_info.dtypes

name               object
id                 object
chr                object
band               object
start_position    float64
end_position      float64
gene_length       float64
dtype: object

In [48]:
_tmp = genes_info[genes_info.isna().any(axis=1)]
display(_tmp)
assert _tmp.shape[0] < 5

,name,id,chr,band,start_position,end_position,gene_length
1585,TMEM133,ENSG00000170647,None,None,NaN,NaN,NaN
2668,TBCE,ENSG00000116957,None,None,NaN,NaN,NaN


In [49]:
genes_info = genes_info.dropna()

In [50]:
genes_info["chr"] = genes_info["chr"].apply(pd.to_numeric, downcast="integer")
genes_info["start_position"] = genes_info["start_position"].astype(int)
genes_info["end_position"] = genes_info["end_position"].astype(int)
genes_info["gene_length"] = genes_info["gene_length"].astype(int)

In [51]:
genes_info.dtypes

name              object
id                object
chr                 int8
band              object
start_position     int64
end_position       int64
gene_length        int64
dtype: object

In [52]:
assert genes_info["name"].is_unique

In [53]:
assert genes_info["id"].is_unique

In [54]:
genes_info.shape

(6449, 7)

In [55]:
genes_info.head()

,name,id,chr,band,start_position,end_position,gene_length
0,MRPL53,ENSG00000204822,2,2p13.1,74471982,74472687,705
1,SNURF,ENSG00000273173,15,15q11.2,24954986,24977850,22864
2,IQSEC1,ENSG00000144711,3,3p25.1,12897043,13283281,386238
3,HBG2,ENSG00000196565,11,11p15.4,5253188,5505605,252417
4,SPRED1,ENSG00000166068,15,15q14,38252836,38357249,104413


In [56]:
genes_info.sort_values("chr")

,name,id,chr,band,start_position,end_position,gene_length
3226,EXOSC10,ENSG00000171824,1,1p36.22,11066618,11099869,33251
510,LPAR3,ENSG00000171517,1,1p22.3,84811602,84893206,81604
511,DVL1,ENSG00000107404,1,1p36.33,1335276,1349418,14142
3942,ASPM,ENSG00000066279,1,1q31.3,197084127,197146694,62567
3940,GALNT2,ENSG00000143641,1,1q42.13,230057990,230282122,224132
...,...,...,...,...,...,...,...
1488,PICK1,ENSG00000100151,22,22q13.1,38056311,38075701,19390
5024,THOC5,ENSG00000100296,22,22q12.2,29505879,29555216,49337
5668,ARSA,ENSG00000100299,22,22q13.33,50622754,50628173,5419
5006,PI4KA,ENSG00000241973,22,22q11.21,20707691,20859417,151726


### Save

In [57]:
genes_info.to_pickle(OUTPUT_DIR_BASE / "genes_info.pkl")

## S-PrediXcan results

### Load results across all tissues

In [58]:
spredixcan_result_files = {
    t: SPREDIXCAN_FOLDER / SPREDIXCAN_FILE_PATTERN.format(tissue=t)
    for t in prediction_model_tissues
}

In [59]:
assert len(spredixcan_result_files) == len(prediction_model_tissues)
display(list(spredixcan_result_files.values())[:5])

[PosixPath('/home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/twas/spredixcan/random.pheno0-gtex_v8-mashr-Thyroid.csv'),
 PosixPath('/home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/twas/spredixcan/random.pheno0-gtex_v8-mashr-Artery_Aorta.csv'),
 PosixPath('/home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/twas/spredixcan/random.pheno0-gtex_v8-mashr-Heart_Atrial_Appendage.csv'),
 PosixPath('/home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/twas/spredixcan/random.pheno0-gtex_v8-mashr-Liver.csv'),
 PosixPath('/home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/null_sims/twas/spredixcan/random.pheno0-gtex_v8-mashr-Heart_Left_Ventricle.csv')]

In [60]:
# look at the structure of one result
pd.read_csv(spredixcan_result_files["Whole_Blood"]).head()

,gene,gene_name,zscore,effect_size,pvalue,var_g,pred_perf_r2,pred_perf_pval,pred_perf_qval,n_snps_used,n_snps_in_cov,n_snps_in_model,best_gwas_p,largest_weight
0,ENSG00000125910.5,S1PR4,-3.949035,NaN,0.000078,0.000104,NaN,NaN,NaN,2,2.0,2,0.000041,0.031808
1,ENSG00000273759.1,RP4-563E14.1,3.825917,NaN,0.000130,0.065132,NaN,NaN,NaN,2,2.0,2,0.000730,0.604017
2,ENSG00000163923.9,RPL39L,-3.749734,NaN,0.000177,0.000004,NaN,NaN,NaN,1,1.0,1,0.000177,0.005621
3,ENSG00000221995.5,TIAF1,-3.683257,NaN,0.000230,0.024214,NaN,NaN,NaN,3,3.0,3,0.001780,0.121188
4,ENSG00000181788.3,SIAH2,-3.564945,NaN,0.000364,0.001093,NaN,NaN,NaN,1,1.0,1,0.000364,0.165562


In [61]:
assert all(f.exists() for f in spredixcan_result_files.values())

In [62]:
spredixcan_dfs = [
    pd.read_csv(
        f,
        usecols=[
            "gene",
            "zscore",
            "pvalue",
            "n_snps_used",
            "n_snps_in_model",
        ],
    )
    .dropna(subset=["gene", "zscore", "pvalue"])
    .assign(tissue=t)
    for t, f in spredixcan_result_files.items()
]

In [63]:
assert len(spredixcan_dfs) == len(prediction_model_tissues)

In [64]:
spredixcan_dfs = pd.concat(spredixcan_dfs)

In [65]:
assert spredixcan_dfs["tissue"].unique().shape[0] == len(prediction_model_tissues)

In [66]:
spredixcan_dfs.shape

(683297, 6)

In [67]:
spredixcan_dfs = spredixcan_dfs.assign(
    gene_id=spredixcan_dfs["gene"].apply(lambda g: g.split(".")[0])
)

In [68]:
spredixcan_dfs.head()

,gene,zscore,pvalue,n_snps_used,n_snps_in_model,tissue,gene_id
0,ENSG00000168830.7,3.988471,0.000067,2,2,Thyroid,ENSG00000168830
1,ENSG00000197566.9,3.881839,0.000104,2,2,Thyroid,ENSG00000197566
2,ENSG00000244274.7,-3.591667,0.000329,2,2,Thyroid,ENSG00000244274
3,ENSG00000139531.12,3.516439,0.000437,4,4,Thyroid,ENSG00000139531
4,ENSG00000172663.8,-3.443800,0.000574,1,1,Thyroid,ENSG00000172663


In [69]:
# leave only common genes
spredixcan_dfs = spredixcan_dfs[spredixcan_dfs["gene_id"].isin(set(genes_info["id"]))]

In [70]:
spredixcan_dfs.shape

(243376, 7)

### Count number of tissues available per gene

In [71]:
spredixcan_genes_n_models = spredixcan_dfs.groupby("gene_id")["tissue"].nunique()

In [72]:
spredixcan_genes_n_models

gene_id
ENSG00000000419     3
ENSG00000000938    36
ENSG00000000971    34
ENSG00000001084    33
ENSG00000001167    40
                   ..
ENSG00000278540    36
ENSG00000278828     4
ENSG00000278845    49
ENSG00000281005    49
ENSG00000282608    10
Name: tissue, Length: 6449, dtype: int64

In [73]:
# testing that in S-MultiXcan I get the same number of tissues per gene
_tmp_smultixcan_results_n_models = (
    smultixcan_results.set_index("gene_id")["n"].astype(int).rename("tissue")
)

_cg = _tmp_smultixcan_results_n_models.index.intersection(
    spredixcan_genes_n_models.index
)
_tmp_smultixcan_results_n_models = _tmp_smultixcan_results_n_models.loc[_cg]
_spredixcan = spredixcan_genes_n_models.loc[_cg]

assert _spredixcan.shape[0] == _tmp_smultixcan_results_n_models.shape[0]
assert _spredixcan.equals(_tmp_smultixcan_results_n_models.loc[_spredixcan.index])

### Get tissues available per gene

In [74]:
spredixcan_genes_models = spredixcan_dfs.groupby("gene_id")["tissue"].apply(
    lambda x: frozenset(x.tolist())
)

In [75]:
spredixcan_genes_models

gene_id
ENSG00000000419    (Brain_Hypothalamus, Cells_Cultured_fibroblast...
ENSG00000000938    (Testis, Brain_Hypothalamus, Pituitary, Adipos...
ENSG00000000971    (Testis, Kidney_Cortex, Ovary, Pituitary, Cell...
ENSG00000001084    (Testis, Pituitary, Cells_Cultured_fibroblasts...
ENSG00000001167    (Testis, Brain_Hypothalamus, Vagina, Kidney_Co...
                                         ...                        
ENSG00000278540    (Brain_Hypothalamus, Vagina, Ovary, Pituitary,...
ENSG00000278828    (Adipose_Visceral_Omentum, Cells_EBV-transform...
ENSG00000278845    (Testis, Brain_Hypothalamus, Vagina, Kidney_Co...
ENSG00000281005    (Testis, Brain_Hypothalamus, Vagina, Kidney_Co...
ENSG00000282608    (Pancreas, Artery_Aorta, Spleen, Adrenal_Gland...
Name: tissue, Length: 6449, dtype: object

In [76]:
assert spredixcan_genes_n_models.shape[0] == spredixcan_genes_models.shape[0]

In [77]:
assert spredixcan_genes_n_models.index.equals(spredixcan_genes_models.index)

In [78]:
assert (spredixcan_genes_models.apply(len) <= len(prediction_model_tissues)).all()

In [79]:
spredixcan_genes_models.apply(len).describe()

count    6449.000000
mean       37.738564
std        12.511355
min         1.000000
25%        32.000000
50%        43.000000
75%        47.000000
max        49.000000
Name: tissue, dtype: float64

In [80]:
# testing that I obtained the right number of tissues
assert (
    spredixcan_genes_models.loc[spredixcan_genes_n_models.index]
    .apply(len)
    .equals(spredixcan_genes_n_models)
)

### Add gene name and set index

In [81]:
spredixcan_genes_models = spredixcan_genes_models.to_frame().reset_index()

In [82]:
spredixcan_genes_models.head()

,gene_id,tissue
0,ENSG00000000419,"(Brain_Hypothalamus, Cells_Cultured_fibroblast..."
1,ENSG00000000938,"(Testis, Brain_Hypothalamus, Pituitary, Adipos..."
2,ENSG00000000971,"(Testis, Kidney_Cortex, Ovary, Pituitary, Cell..."
3,ENSG00000001084,"(Testis, Pituitary, Cells_Cultured_fibroblasts..."
4,ENSG00000001167,"(Testis, Brain_Hypothalamus, Vagina, Kidney_Co..."


In [83]:
spredixcan_genes_models = spredixcan_genes_models.assign(
    gene_name=spredixcan_genes_models["gene_id"].apply(
        lambda g: Gene.GENE_ID_TO_NAME_MAP[g]
    )
)

In [84]:
spredixcan_genes_models = spredixcan_genes_models[["gene_id", "gene_name", "tissue"]]

In [85]:
spredixcan_genes_models = spredixcan_genes_models.set_index("gene_id")

In [86]:
spredixcan_genes_models.head()

,gene_name,tissue
gene_id,,
ENSG00000000419,DPM1,"(Brain_Hypothalamus, Cells_Cultured_fibroblast..."
ENSG00000000938,FGR,"(Testis, Brain_Hypothalamus, Pituitary, Adipos..."
ENSG00000000971,CFH,"(Testis, Kidney_Cortex, Ovary, Pituitary, Cell..."
ENSG00000001084,GCLC,"(Testis, Pituitary, Cells_Cultured_fibroblasts..."
ENSG00000001167,NFYA,"(Testis, Brain_Hypothalamus, Vagina, Kidney_Co..."


### Add number of tissues

In [87]:
spredixcan_genes_models = spredixcan_genes_models.assign(
    n_tissues=spredixcan_genes_models["tissue"].apply(len)
)

In [88]:
spredixcan_genes_models.head()

,gene_name,tissue,n_tissues
gene_id,,,
ENSG00000000419,DPM1,"(Brain_Hypothalamus, Cells_Cultured_fibroblast...",3
ENSG00000000938,FGR,"(Testis, Brain_Hypothalamus, Pituitary, Adipos...",36
ENSG00000000971,CFH,"(Testis, Kidney_Cortex, Ovary, Pituitary, Cell...",34
ENSG00000001084,GCLC,"(Testis, Pituitary, Cells_Cultured_fibroblasts...",33
ENSG00000001167,NFYA,"(Testis, Brain_Hypothalamus, Vagina, Kidney_Co...",40


### Save

Here I quickly save these results to a file, given that the next steps (covariates) are slow to compute.

In [89]:
# this is important, other scripts depend on gene_name to be unique
assert spredixcan_genes_models["gene_name"].is_unique

In [90]:
assert not spredixcan_genes_models.isna().any(None)

/tmp/ipykernel_1282128/1381297071.py:1: FutureWarning: In a future version of pandas all arguments of DataFrame.any and Series.any will be keyword-only.
  assert not spredixcan_genes_models.isna().any(None)


In [91]:
spredixcan_genes_models.to_pickle(OUTPUT_DIR_BASE / "gene_tissues.pkl")

## Add covariates based on S-PrediXcan results

This extend the previous file with more columns

### Get gene's objects

In [92]:
spredixcan_gene_obj = {
    gene_id: Gene(ensembl_id=gene_id) for gene_id in spredixcan_genes_models.index
}

In [93]:
len(spredixcan_gene_obj)

6449

### Add genes' variance captured by principal components

In [94]:
def _get_gene_pc_variance(gene_row):
    gene_id = gene_row.name
    gene_tissues = gene_row["tissue"]
    gene_obj = spredixcan_gene_obj[gene_id]

    u, s, vt = gene_obj.get_tissues_correlations_svd(
        tissues=gene_tissues,
        snps_subset=gwas_variants_ids_set,
        reference_panel=REFERENCE_PANEL,
        model_type=EQTL_MODEL,
        # use_covariance_matrix=True,
    )

    return s

In [95]:
_tmp = spredixcan_genes_models.loc["ENSG00000188976"]
_get_gene_pc_variance(_tmp)

array([35.60509556,  3.87543604,  2.16044422,  1.41758395,  1.26548868])

In [96]:
spredixcan_genes_tissues_pc_variance = spredixcan_genes_models.apply(
    _get_gene_pc_variance, axis=1
)

In [97]:
spredixcan_genes_tissues_pc_variance

gene_id
ENSG00000000419    [1.052540658399403, 1.024362410165966, 0.92309...
ENSG00000000938    [31.63266933637128, 2.078367751851144, 1.27161...
ENSG00000000971    [21.56076992568329, 7.310170987045466, 1.83778...
ENSG00000001084    [21.747365896432562, 4.700645786683649, 2.2499...
ENSG00000001167                                  [38.28994327673915]
                                         ...                        
ENSG00000278540    [30.239540272954734, 3.044832646579891, 1.6793...
ENSG00000278828              [3.062182335109643, 0.9378176648903567]
ENSG00000278845                [45.6143801458816, 2.270555362800341]
ENSG00000281005                                  [48.30170739785573]
ENSG00000282608    [4.6920673526712395, 2.5190991331399073, 1.013...
Length: 6449, dtype: object

In [98]:
# # testing
# assert spredixcan_genes_tissues_pc_variance.loc[
#     "ENSG00000188976"
# ].sum() == pytest.approx(44.01605629086847)
# # this is using the covariance:
# # assert spredixcan_genes_tissues_pc_variance.loc["ENSG00000188976"].sum() == pytest.approx(1.1492946006449425)

In [99]:
# add to spredixcan_genes_models
spredixcan_genes_models = spredixcan_genes_models.join(
    spredixcan_genes_tissues_pc_variance.rename("tissues_pc_variances")
)

In [100]:
spredixcan_genes_models.shape

(6449, 4)

In [101]:
spredixcan_genes_models.head()

,gene_name,tissue,n_tissues,tissues_pc_variances
gene_id,,,,
ENSG00000000419,DPM1,"(Brain_Hypothalamus, Cells_Cultured_fibroblast...",3,"[1.052540658399403, 1.024362410165966, 0.92309..."
ENSG00000000938,FGR,"(Testis, Brain_Hypothalamus, Pituitary, Adipos...",36,"[31.63266933637128, 2.078367751851144, 1.27161..."
ENSG00000000971,CFH,"(Testis, Kidney_Cortex, Ovary, Pituitary, Cell...",34,"[21.56076992568329, 7.310170987045466, 1.83778..."
ENSG00000001084,GCLC,"(Testis, Pituitary, Cells_Cultured_fibroblasts...",33,"[21.747365896432562, 4.700645786683649, 2.2499..."
ENSG00000001167,NFYA,"(Testis, Brain_Hypothalamus, Vagina, Kidney_Co...",40,[38.28994327673915]


### Add gene variance per tissue

In [102]:
def _get_gene_variances(gene_row):
    gene_id = gene_row.name
    gene_tissues = gene_row["tissue"]

    tissue_variances = {}
    gene_obj = spredixcan_gene_obj[gene_id]

    for tissue in gene_tissues:
        tissue_var = gene_obj.get_pred_expression_variance(
            tissue=tissue,
            reference_panel=REFERENCE_PANEL,
            model_type=EQTL_MODEL,
            snps_subset=gwas_variants_ids_set,
        )

        if tissue_var is not None:
            tissue_variances[tissue] = tissue_var

    return tissue_variances

In [103]:
_tmp = spredixcan_genes_models.loc["ENSG00000000419"]
_get_gene_variances(_tmp)

{'Brain_Hypothalamus': 0.013162153504206677,
 'Cells_Cultured_fibroblasts': 0.0030618005265686445,
 'Brain_Substantia_nigra': 0.0004867380334260178}

In [104]:
spredixcan_genes_tissues_variance = spredixcan_genes_models.apply(
    _get_gene_variances, axis=1
)

In [105]:
spredixcan_genes_tissues_variance

gene_id
ENSG00000000419    {'Brain_Hypothalamus': 0.013162153504206677, '...
ENSG00000000938    {'Testis': 0.006332646674623576, 'Brain_Hypoth...
ENSG00000000971    {'Testis': 0.004542368000299982, 'Kidney_Corte...
ENSG00000001084    {'Testis': 0.0163989833924462, 'Pituitary': 0....
ENSG00000001167    {'Testis': 0.058424526236390065, 'Brain_Hypoth...
                                         ...                        
ENSG00000278540    {'Brain_Hypothalamus': 0.0037182047904750328, ...
ENSG00000278828    {'Adipose_Visceral_Omentum': 0.000128291681544...
ENSG00000278845    {'Testis': 0.02168997277314074, 'Brain_Hypotha...
ENSG00000281005    {'Testis': 0.11707441085667628, 'Brain_Hypotha...
ENSG00000282608    {'Pancreas': 0.009723795909827731, 'Artery_Aor...
Length: 6449, dtype: object

In [106]:
# # testing
# _gene_id = "ENSG00000188976"
# x = spredixcan_genes_tissues_variance.loc[_gene_id]
# # expected value obtained by sum of PCA eigenvalues on this gene's predicted expression
# assert np.sum(list(x.values())) == pytest.approx(1.2326202607409493)

In [107]:
# testing
spredixcan_genes_tissues_variance.loc["ENSG00000000419"]

{'Brain_Hypothalamus': 0.013162153504206677,
 'Cells_Cultured_fibroblasts': 0.0030618005265686445,
 'Brain_Substantia_nigra': 0.0004867380334260178}

In [108]:
# add to spredixcan_genes_models
spredixcan_genes_models = spredixcan_genes_models.join(
    spredixcan_genes_tissues_variance.rename("tissues_variances")
)

In [109]:
spredixcan_genes_models.shape

(6449, 5)

In [110]:
spredixcan_genes_models.head()

,gene_name,tissue,n_tissues,tissues_pc_variances,tissues_variances
gene_id,,,,,
ENSG00000000419,DPM1,"(Brain_Hypothalamus, Cells_Cultured_fibroblast...",3,"[1.052540658399403, 1.024362410165966, 0.92309...","{'Brain_Hypothalamus': 0.013162153504206677, '..."
ENSG00000000938,FGR,"(Testis, Brain_Hypothalamus, Pituitary, Adipos...",36,"[31.63266933637128, 2.078367751851144, 1.27161...","{'Testis': 0.006332646674623576, 'Brain_Hypoth..."
ENSG00000000971,CFH,"(Testis, Kidney_Cortex, Ovary, Pituitary, Cell...",34,"[21.56076992568329, 7.310170987045466, 1.83778...","{'Testis': 0.004542368000299982, 'Kidney_Corte..."
ENSG00000001084,GCLC,"(Testis, Pituitary, Cells_Cultured_fibroblasts...",33,"[21.747365896432562, 4.700645786683649, 2.2499...","{'Testis': 0.0163989833924462, 'Pituitary': 0...."
ENSG00000001167,NFYA,"(Testis, Brain_Hypothalamus, Vagina, Kidney_Co...",40,[38.28994327673915],"{'Testis': 0.058424526236390065, 'Brain_Hypoth..."


### Count number of SNPs predictors used across tissue models

In [111]:
spredixcan_genes_sum_of_n_snps_used = (
    spredixcan_dfs.groupby("gene_id")["n_snps_used"].sum().rename("n_snps_used_sum")
)

In [112]:
spredixcan_genes_sum_of_n_snps_used

gene_id
ENSG00000000419     3
ENSG00000000938    40
ENSG00000000971    44
ENSG00000001084    47
ENSG00000001167    48
                   ..
ENSG00000278540    44
ENSG00000278828     5
ENSG00000278845    91
ENSG00000281005    81
ENSG00000282608    12
Name: n_snps_used_sum, Length: 6449, dtype: int64

In [113]:
# add sum of snps used to spredixcan_genes_models
spredixcan_genes_models = spredixcan_genes_models.join(
    spredixcan_genes_sum_of_n_snps_used
)

In [114]:
spredixcan_genes_models.shape

(6449, 6)

In [115]:
spredixcan_genes_models.head()

,gene_name,tissue,n_tissues,tissues_pc_variances,tissues_variances,n_snps_used_sum
gene_id,,,,,,
ENSG00000000419,DPM1,"(Brain_Hypothalamus, Cells_Cultured_fibroblast...",3,"[1.052540658399403, 1.024362410165966, 0.92309...","{'Brain_Hypothalamus': 0.013162153504206677, '...",3
ENSG00000000938,FGR,"(Testis, Brain_Hypothalamus, Pituitary, Adipos...",36,"[31.63266933637128, 2.078367751851144, 1.27161...","{'Testis': 0.006332646674623576, 'Brain_Hypoth...",40
ENSG00000000971,CFH,"(Testis, Kidney_Cortex, Ovary, Pituitary, Cell...",34,"[21.56076992568329, 7.310170987045466, 1.83778...","{'Testis': 0.004542368000299982, 'Kidney_Corte...",44
ENSG00000001084,GCLC,"(Testis, Pituitary, Cells_Cultured_fibroblasts...",33,"[21.747365896432562, 4.700645786683649, 2.2499...","{'Testis': 0.0163989833924462, 'Pituitary': 0....",47
ENSG00000001167,NFYA,"(Testis, Brain_Hypothalamus, Vagina, Kidney_Co...",40,[38.28994327673915],"{'Testis': 0.058424526236390065, 'Brain_Hypoth...",48


### Count number of SNPs predictors in models across tissue models

In [116]:
spredixcan_genes_sum_of_n_snps_in_model = (
    spredixcan_dfs.groupby("gene_id")["n_snps_in_model"]
    .sum()
    .rename("n_snps_in_model_sum")
)

In [117]:
spredixcan_genes_sum_of_n_snps_in_model

gene_id
ENSG00000000419     3
ENSG00000000938    40
ENSG00000000971    44
ENSG00000001084    47
ENSG00000001167    48
                   ..
ENSG00000278540    44
ENSG00000278828     5
ENSG00000278845    91
ENSG00000281005    81
ENSG00000282608    12
Name: n_snps_in_model_sum, Length: 6449, dtype: int64

In [118]:
# add sum of snps in model to spredixcan_genes_models
spredixcan_genes_models = spredixcan_genes_models.join(
    spredixcan_genes_sum_of_n_snps_in_model
)

In [119]:
spredixcan_genes_models.shape

(6449, 7)

In [120]:
spredixcan_genes_models.head()

,gene_name,tissue,n_tissues,tissues_pc_variances,tissues_variances,n_snps_used_sum,n_snps_in_model_sum
gene_id,,,,,,,
ENSG00000000419,DPM1,"(Brain_Hypothalamus, Cells_Cultured_fibroblast...",3,"[1.052540658399403, 1.024362410165966, 0.92309...","{'Brain_Hypothalamus': 0.013162153504206677, '...",3,3
ENSG00000000938,FGR,"(Testis, Brain_Hypothalamus, Pituitary, Adipos...",36,"[31.63266933637128, 2.078367751851144, 1.27161...","{'Testis': 0.006332646674623576, 'Brain_Hypoth...",40,40
ENSG00000000971,CFH,"(Testis, Kidney_Cortex, Ovary, Pituitary, Cell...",34,"[21.56076992568329, 7.310170987045466, 1.83778...","{'Testis': 0.004542368000299982, 'Kidney_Corte...",44,44
ENSG00000001084,GCLC,"(Testis, Pituitary, Cells_Cultured_fibroblasts...",33,"[21.747365896432562, 4.700645786683649, 2.2499...","{'Testis': 0.0163989833924462, 'Pituitary': 0....",47,47
ENSG00000001167,NFYA,"(Testis, Brain_Hypothalamus, Vagina, Kidney_Co...",40,[38.28994327673915],"{'Testis': 0.058424526236390065, 'Brain_Hypoth...",48,48


### Summarize prediction models for each gene

In [121]:
def _summarize_gene_models(gene_id):
    """
    For a given gene ID, it returns a dataframe with predictor SNPs in rows and tissues in columns, where
    values are the weights of SNPs in those tissues.
    It can contain NaNs.
    """
    gene_obj = spredixcan_gene_obj[gene_id]
    gene_tissues = spredixcan_genes_models.loc[gene_id, "tissue"]

    gene_models = {}
    gene_unique_snps = set()
    for t in gene_tissues:
        gene_model = gene_obj.get_prediction_weights(tissue=t, model_type=EQTL_MODEL)
        gene_models[t] = gene_model

        gene_unique_snps.update(set(gene_model.index))

    df = pd.DataFrame(
        data=np.nan, index=list(gene_unique_snps), columns=list(gene_tissues)
    )

    for t in df.columns:
        for snp in df.index:
            gene_model = gene_models[t]

            if snp in gene_model.index:
                df.loc[snp, t] = gene_model.loc[snp]

    return df

In [122]:
# testing
spredixcan_gene_obj["ENSG00000000419"].get_prediction_weights(
    tissue="Brain_Hypothalamus", model_type=EQTL_MODEL
)

varID
chr20_50862947_C_T_b38    0.431375
Name: weight, dtype: float64

In [123]:
spredixcan_gene_obj["ENSG00000000419"].get_prediction_weights(
    tissue="Brain_Substantia_nigra", model_type=EQTL_MODEL
)

varID
chr20_50957480_C_T_b38   -0.146796
Name: weight, dtype: float64

In [124]:
# # testing
# _gene_id = "ENSG00000000419"

# _gene_model = _summarize_gene_models(_gene_id)
# assert (
#     _gene_model.loc["chr20_50862947_C_T_b38", "Brain_Hypothalamus"].round(5) == 0.43138
# )
# assert pd.isnull(_gene_model.loc["chr20_50957480_C_T_b38", "Brain_Hypothalamus"])

# assert pd.isnull(_gene_model.loc["chr20_50862947_C_T_b38", "Brain_Substantia_nigra"])
# assert (
#     _gene_model.loc["chr20_50957480_C_T_b38", "Brain_Substantia_nigra"].round(5)
#     == -0.1468
# )

In [125]:
gene_models = {}

for gene_id in spredixcan_genes_models.index:
    gene_models[gene_id] = _summarize_gene_models(gene_id)

In [126]:
# # testing
# _gene_id = "ENSG00000000419"

# _gene_model = gene_models[_gene_id]
# assert (
#     _gene_model.loc["chr20_50862947_C_T_b38", "Brain_Hypothalamus"].round(5) == 0.43138
# )
# assert pd.isnull(_gene_model.loc["chr20_50957480_C_T_b38", "Brain_Hypothalamus"])

# assert pd.isnull(_gene_model.loc["chr20_50862947_C_T_b38", "Brain_Substantia_nigra"])
# assert (
#     _gene_model.loc["chr20_50957480_C_T_b38", "Brain_Substantia_nigra"].round(5)
#     == -0.1468
# )

In [127]:
# save
import gzip

with gzip.GzipFile(OUTPUT_DIR_BASE / "gene_tissues_models.pkl.gz", "w") as f:
    pickle.dump(gene_models, f)

In [128]:
# testing saved file
with gzip.GzipFile(OUTPUT_DIR_BASE / "gene_tissues_models.pkl.gz", "r") as f:
    _tmp = pickle.load(f)

In [129]:
assert len(gene_models) == len(_tmp)
assert gene_models["ENSG00000000419"].equals(_tmp["ENSG00000000419"])

### Count number of _unique_ SNPs predictors used and available across tissue models

In [130]:
def _count_unique_snps(gene_id):
    """
    For a gene_id, it counts unique SNPs in all models and their intersection with GWAS SNPs (therefore, used by S-PrediXcan).
    """
    gene_tissues = spredixcan_genes_models.loc[gene_id, "tissue"]

    gene_unique_snps = set()
    for t in gene_tissues:
        t_snps = set(gene_models[gene_id].index)
        gene_unique_snps.update(t_snps)

    gene_unique_snps_in_gwas = gwas_variants_ids_set.intersection(gene_unique_snps)

    return pd.Series(
        {
            "unique_n_snps_in_model": len(gene_unique_snps),
            "unique_n_snps_used": len(gene_unique_snps_in_gwas),
        }
    )

In [131]:
# testing
spredixcan_genes_models[spredixcan_genes_models["n_snps_used_sum"] == 2].head()

,gene_name,tissue,n_tissues,tissues_pc_variances,tissues_variances,n_snps_used_sum,n_snps_in_model_sum
gene_id,,,,,,,
ENSG00000010256,UQCRC1,"(Thyroid, Whole_Blood)",2,"[1.1048350039574655, 0.8951649960425345]","{'Thyroid': 0.0008025245652862433, 'Whole_Bloo...",2,2
ENSG00000017427,IGF1,"(Testis, Brain_Amygdala)",2,"[1.0389319526404415, 0.9610680473595585]","{'Testis': 0.003862056066316527, 'Brain_Amygda...",2,2
ENSG00000043093,DCUN1D1,"(Esophagus_Gastroesophageal_Junction, Esophagu...",2,"[1.4951653523020276, 0.5048346476979725]",{'Esophagus_Gastroesophageal_Junction': 0.0002...,2,2
ENSG00000081377,CDC14B,"(Muscle_Skeletal, Brain_Nucleus_accumbens_basa...",2,"[1.1285007973441137, 0.8714992026558863]","{'Muscle_Skeletal': 0.0076846621297156385, 'Br...",2,2
ENSG00000104856,RELB,"(Brain_Cerebellar_Hemisphere, Esophagus_Gastro...",2,"[1.065634630043069, 0.9343653699569308]",{'Brain_Cerebellar_Hemisphere': 0.006361992573...,2,2


In [132]:
# case with two snps, not repeated across tissues
_gene_id = "ENSG00000000419"
display(
    spredixcan_gene_obj[_gene_id].get_prediction_weights(
        tissue="Brain_Hypothalamus", model_type=EQTL_MODEL
    )
)
display(
    spredixcan_gene_obj[_gene_id].get_prediction_weights(
        tissue="Brain_Substantia_nigra", model_type=EQTL_MODEL
    )
)

varID
chr20_50862947_C_T_b38    0.431375
Name: weight, dtype: float64

varID
chr20_50957480_C_T_b38   -0.146796
Name: weight, dtype: float64

In [133]:
# _tmp = _count_unique_snps(_gene_id)
# assert _tmp.shape[0] == 2
# assert _tmp["unique_n_snps_in_model"] == 2
# assert _tmp["unique_n_snps_used"] == 2

In [134]:
# get unique snps for all genes
spredixcan_genes_unique_n_snps = spredixcan_genes_models.groupby("gene_id").apply(
    lambda x: _count_unique_snps(x.name)
)

In [135]:
spredixcan_genes_unique_n_snps.head()

,unique_n_snps_in_model,unique_n_snps_used
gene_id,,
ENSG00000000419,3,3
ENSG00000000938,5,5
ENSG00000000971,12,12
ENSG00000001084,24,24
ENSG00000001167,14,14


In [136]:
assert (
    spredixcan_genes_unique_n_snps["unique_n_snps_in_model"]
    >= spredixcan_genes_unique_n_snps["unique_n_snps_used"]
).all()

In [137]:
# add unique snps to spredixcan_genes_models
spredixcan_genes_models = spredixcan_genes_models.join(spredixcan_genes_unique_n_snps)

In [138]:
spredixcan_genes_models.shape

(6449, 9)

In [139]:
spredixcan_genes_models.head()

,gene_name,tissue,n_tissues,tissues_pc_variances,tissues_variances,n_snps_used_sum,n_snps_in_model_sum,unique_n_snps_in_model,unique_n_snps_used
gene_id,,,,,,,,,
ENSG00000000419,DPM1,"(Brain_Hypothalamus, Cells_Cultured_fibroblast...",3,"[1.052540658399403, 1.024362410165966, 0.92309...","{'Brain_Hypothalamus': 0.013162153504206677, '...",3,3,3,3
ENSG00000000938,FGR,"(Testis, Brain_Hypothalamus, Pituitary, Adipos...",36,"[31.63266933637128, 2.078367751851144, 1.27161...","{'Testis': 0.006332646674623576, 'Brain_Hypoth...",40,40,5,5
ENSG00000000971,CFH,"(Testis, Kidney_Cortex, Ovary, Pituitary, Cell...",34,"[21.56076992568329, 7.310170987045466, 1.83778...","{'Testis': 0.004542368000299982, 'Kidney_Corte...",44,44,12,12
ENSG00000001084,GCLC,"(Testis, Pituitary, Cells_Cultured_fibroblasts...",33,"[21.747365896432562, 4.700645786683649, 2.2499...","{'Testis': 0.0163989833924462, 'Pituitary': 0....",47,47,24,24
ENSG00000001167,NFYA,"(Testis, Brain_Hypothalamus, Vagina, Kidney_Co...",40,[38.28994327673915],"{'Testis': 0.058424526236390065, 'Brain_Hypoth...",48,48,14,14


### Save

In [140]:
# this is important, other scripts depend on gene_name to be unique
assert spredixcan_genes_models["gene_name"].is_unique

In [141]:
assert not spredixcan_genes_models.isna().any(None)

/tmp/ipykernel_1282128/1381297071.py:1: FutureWarning: In a future version of pandas all arguments of DataFrame.any and Series.any will be keyword-only.
  assert not spredixcan_genes_models.isna().any(None)


In [142]:
spredixcan_genes_models.to_pickle(OUTPUT_DIR_BASE / "gene_tissues.pkl")